# Task 2: Visual Embedding Generation

**Mục tiêu:** So sánh OpenCLIP (ViT-H-14, BEiT-3) hiện tại với SOTA VLM (InternVideo2, Qwen2-VL-7B lượng tử hóa 4-bit).

Notebook này được thiết kế để chạy trên Kaggle GPU T4 (16GB VRAM).

In [ ]:
# 1. Cài đặt các thư viện cần thiết
!pip install torch torchvision open_clip_torch transformers accelerate bitsandbytes

In [ ]:
import torch
import open_clip
from PIL import Image
import time

# --- P1: Baseline (OpenCLIP ViT-H-14 như hệ thống hiện tại) ---
print('Loading OpenCLIP...')
model, _, preprocess = open_clip.create_model_and_transforms('ViT-H-14-quickgelu', pretrained='dfn5b', device='cuda')

image = Image.new('RGB', (224, 224), color = 'red') # Ảnh giả lập
image_input = preprocess(image).unsqueeze(0).to('cuda')

start_time = time.time()
with torch.no_grad():
    image_features = model.encode_image(image_input)
    image_features /= image_features.norm(dim=-1, keepdim=True)
end_time = time.time()

print(f'OpenCLIP Inference Time: {end_time - start_time:.4f}s')
print(f'Feature shape: {image_features.shape}')
del model, image_input
torch.cuda.empty_cache()

In [ ]:
# --- P2: SOTA (Qwen2-VL-7B Quantized 4-bit) ---
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from transformers import BitsAndBytesConfig

print('Loading Qwen2-VL-7B (4-bit)...')
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

try:
    processor = AutoProcessor.from_pretrained('Qwen/Qwen2-VL-7B-Instruct')
    model_qwen = Qwen2VLForConditionalGeneration.from_pretrained(
        'Qwen/Qwen2-VL-7B-Instruct', 
        device_map='auto', 
        quantization_config=quantization_config
    )
    
    messages = [
        {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": "Describe this image."}]}
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt").to('cuda')
    
    start_time = time.time()
    with torch.no_grad():
        generated_ids = model_qwen.generate(**inputs, max_new_tokens=50)
    end_time = time.time()
    
    print(f'Qwen2-VL Inference Time: {end_time - start_time:.4f}s')
except Exception as e:
    print('Lỗi tải model Qwen2-VL:', e)

## Đánh giá:
- OpenCLIP sinh vector nhúng rất nhanh (miliseconds), hoàn hảo cho Faiss Index.
- Qwen2-VL-7B tốn nhiều thời gian hơn (seconds) nhưng hiểu hình ảnh ở cấp độ VLM (sinh ra text mô tả phức tạp). Hệ thống tương lai có thể dùng OpenCLIP để truy xuất thô (Retrieval) và Qwen2-VL để đánh giá lại (Reranking).